# Load YOLO segmentation datasets with compact masks

CPU demonstration using a local Supervision checkout. Open this notebook with the kernel working directory set to `examples/compact_mask` in that checkout. No model, Roboflow account, or dataset download is required.

The setup installs Supervision from the checkout so the library and the adjacent `benchmark_yolo.py` use the same revision. Restart the kernel after installation if Supervision was already imported. The benchmark runs the local script without downloading executable code.

The Linux benchmark creates 16 images with 8 small polygons each, compares default dense loading with `compact_masks=True`, and checks exact pixel, box and class-ID equality. Each mode is measured in three fresh processes.

In [ ]:
from pathlib import Path

# Fail before installation if the kernel is outside the checkout's example directory.
_ = Path("benchmark_yolo.py").resolve(strict=True)
_ = Path("../../pyproject.toml").resolve(strict=True)
%pip install -q -e ../..

In [ ]:
import subprocess
import sys
from pathlib import Path

# Fresh processes keep mask decoding out of the measured peak RSS.
script = Path("benchmark_yolo.py").resolve(strict=True)
subprocess.run(  # noqa: S603
    [sys.executable, str(script)], check=True, timeout=120
)

## Interpret the output

Compare dense and compact samples within each scenario to separate retained mask storage from process overhead. The separate equality check confirms that reduced storage preserves the loaded annotations; timing and RSS depend on the Linux host and are not performance guarantees.

- `mask_buffer_bytes` counts dense array bytes or compact RLE/crop-shape/offset array bytes, excluding Python-object and allocator overhead.
- `peak_rss_mib` is Linux process peak RSS, including imports and input generation; it is a different measurement from retained mask storage.
- `load_seconds` excludes imports and input generation. Pixel equality runs separately so decoding does not inflate measured peak RSS.
- These sparse synthetic polygons compress well; fragmented masks can compress poorly. This is compact eager loading: each polygon still temporarily uses a full-frame mask. Calling `to_dense()` allocates dense masks again.

Related work: [lazy annotations #1484](https://github.com/roboflow/supervision/pull/1484), [compact ingestion #2367](https://github.com/roboflow/supervision/pull/2367), [polygon crop bounds #2678](https://github.com/roboflow/supervision/pull/2678), and [RF-DETR lazy loading #851](https://github.com/roboflow/rf-detr/pull/851). This demo uses the existing CompactMask implementation and does not replace RF-DETR's fully lazy loader.